# Model 5 Training — Pokemon Showdown Policy

**Generation lineage:**
- model2 — joint action + predict_turn_outcome  
- model3 — same, re-run with improved data  
- model4 — model3 + predict_value  
- **model5 — model4 + predict_turn_sequence** ← this run

**New in model5:** LSTM sequence head that learns to predict the ordered event sequence  
(moves, damage, switches, etc.) for each turn, conditioned on state + both actions.  
Vocab size ~600+ composite tokens. Sequence loss starts ~6.4 (log vocab) and decreases.

**Runtime:** Set Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
# ── 1. Verify GPU ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('WARNING: No GPU detected — go to Runtime → Change runtime type → T4 GPU')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF sees:', gpus if gpus else 'NO GPU — check runtime type')

In [ ]:
# ── 2. Clone repo (feature branch with Session 1+2 work) ───────────────────
import os

REPO_URL    = 'https://github.com/AlterProgramming/Pokemon-Showdown-Sim.git'
BRANCH      = 'feature/turn-event-tokenizer-and-sequence-head'
REPO_DIR    = '/content/Pokemon-Showdown-Sim'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!git log --oneline -3

In [ ]:
# ── 3. Install dependencies ────────────────────────────────────────────────
# Colab ships with TF pre-installed. We only add kagglehub + pin keras.
# Avoid reinstalling TF (slow, risks version conflicts).
import tensorflow as tf
print('Colab TF version:', tf.__version__)

!pip install -q kagglehub

# Colab TF 2.15/2.16 ships with keras 2.x; we need keras 3.x for the model.
import importlib
try:
    import keras
    major = int(keras.__version__.split('.')[0])
    if major < 3:
        print(f'keras {keras.__version__} is too old — upgrading to 3.x')
        !pip install -q 'keras>=3.0'
        importlib.invalidate_caches()
    else:
        print(f'keras {keras.__version__} OK')
except ImportError:
    !pip install -q 'keras>=3.0'

# Restart runtime once after first install so keras 3 is visible
print('\nIf this is your first run, use Runtime → Restart session, then re-run from cell 1.')

In [ ]:
# ── 4. Download training data (public — no credentials needed) ─────────────
import kagglehub, os

DATASET = 'thephilliplin/pokemon-showdown-battles-gen9-randbats'
print(f'Downloading {DATASET} ...')
data_path = kagglehub.dataset_download(DATASET)

import glob
json_files = glob.glob(os.path.join(data_path, '*.json'))
print(f'Dataset path : {data_path}')
print(f'JSON files   : {len(json_files):,}')

In [ ]:
# ── 5. Training configuration ──────────────────────────────────────────────
import os
os.chdir('/content/Pokemon-Showdown-Sim')

MODEL_NAME          = 'model_5'
OUTPUT_DIR          = '/content/artifacts'
MAX_BATTLES         = 5000
EPOCHS              = 30
BATCH_SIZE          = 512   # larger batch is more efficient on GPU
LEARNING_RATE       = 0.001
PATIENCE            = 3

# Sequence head (Session 2)
SEQUENCE_WEIGHT     = 0.1
SEQUENCE_HIDDEN_DIM = 128
MAX_SEQ_LEN         = 32

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config ready. Artifacts will be saved to:', OUTPUT_DIR)

In [ ]:
# ── 6. Train ───────────────────────────────────────────────────────────────
import subprocess, sys, os

REPO_DIR = '/content/Pokemon-Showdown-Sim'

# Ensure the repo root is on PYTHONPATH so bare imports like
# `from TurnEventTokenizer import ...` resolve correctly inside the subprocess.
env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + os.pathsep + env.get('PYTHONPATH', '')

cmd = [
    sys.executable, '-u', 'train_policy.py',
    data_path,
    '--model-name',            MODEL_NAME,
    '--output-dir',            OUTPUT_DIR,
    '--max-battles',           str(MAX_BATTLES),
    '--epochs',                str(EPOCHS),
    '--batch-size',            str(BATCH_SIZE),
    '--learning-rate',         str(LEARNING_RATE),
    '--patience',              str(PATIENCE),
    '--include-switches',
    '--predict-turn-outcome',
    '--predict-value',
    '--predict-turn-sequence',
    '--sequence-weight',       str(SEQUENCE_WEIGHT),
    '--sequence-hidden-dim',   str(SEQUENCE_HIDDEN_DIM),
    '--max-seq-len',           str(MAX_SEQ_LEN),
    '--transition-weight',     '0.25',
    '--value-weight',          '0.25',
    '--action-embed-dim',      '32',
    '--transition-hidden-dim', '256',
    '--hidden-dim',            '256',
    '--depth',                 '3',
    '--dropout',               '0.1',
    '--policy-return-weighting',    'exp',
    '--policy-return-weight-scale', '0.75',
    '--policy-return-weight-min',   '0.25',
    '--policy-return-weight-max',   '4.0',
    '--seed',                  '42',
]

print('Running:', ' '.join(cmd[:6]), '...')
print('=' * 60)

# Stream output live; cwd + PYTHONPATH both point at repo root
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, cwd=REPO_DIR, env=env)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('=' * 60)
print('Exit code:', proc.returncode)

In [ ]:
# ── 7. Inspect artifacts ───────────────────────────────────────────────────
import os, json

artifacts = sorted(os.listdir(OUTPUT_DIR))
print('Saved artifacts:')
for f in artifacts:
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f:50s}  {size/1024:.1f} KB')

# Print key metadata fields
meta_path = os.path.join(OUTPUT_DIR, f'training_metadata_{MODEL_NAME}.json')
if not os.path.exists(meta_path):
    # fallback: find any training_metadata file
    candidates = [f for f in artifacts if f.startswith('training_metadata')]
    meta_path = os.path.join(OUTPUT_DIR, candidates[0]) if candidates else None

if meta_path and os.path.exists(meta_path):
    meta = json.loads(open(meta_path).read())
    for k in ('model_name', 'epochs_completed', 'train_examples', 'val_examples',
              'policy_param_count', 'training_param_count',
              'sequence_vocab_size', 'max_seq_len', 'training_regime'):
        if k in meta:
            print(f'{k}: {meta[k]}')

In [ ]:
# ── 8. Download artifacts to local machine ─────────────────────────────────
# Downloads each artifact file to your browser.
# Skip files you don't need (large .keras files are 5–20 MB).
from google.colab import files
import os

DOWNLOAD = [
    f'model_{MODEL_NAME}.keras',
    f'action_vocab_{MODEL_NAME}.json',
    f'action_context_vocab_{MODEL_NAME}.json',
    f'sequence_vocab_{MODEL_NAME}.json',
    f'training_metadata_{MODEL_NAME}.json',
]

for fname in DOWNLOAD:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        print(f'Downloading {fname} ...')
        files.download(fpath)
    else:
        print(f'Not found (may use different name): {fname}')

# Also list everything available if above names don't match
print('\nAll artifacts in output dir:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(' ', f)

In [ ]:
# ── 13. Push artifacts to GCS ─────────────────────────────────────────────
# Uploads trained artifacts to gs://artifacts-model-serving/artifacts/{family}/{run_id}/
# Uses your logged-in Colab Google account — no secrets required.
import os, glob

BUCKET     = 'artifacts-model-serving'
GCS_PREFIX = f'artifacts/vector_joint_bc_transition_value_v1/{RUN_ID}'

try:
    from google.colab import auth
    from google.cloud import storage

    auth.authenticate_user()  # one-time browser prompt per Colab session
    client = storage.Client(project='pokemon-battle-agent-494015')
    bucket = client.bucket(BUCKET)

    pushed = []
    for local_path in glob.glob(os.path.join(OUTPUT_DIR, '*')):
        if not os.path.isfile(local_path):
            continue
        fname     = os.path.basename(local_path)
        blob_name = f'{GCS_PREFIX}/{fname}'
        bucket.blob(blob_name).upload_from_filename(local_path)
        pushed.append(blob_name)
        print(f'  ✓ {blob_name}')

    print(f'\nPushed {len(pushed)} files → gs://{BUCKET}/{GCS_PREFIX}/')
    print('Run /gcs-push locally to sync model_registry.json.')

except Exception as e:
    print(f'GCS push failed: {e}')
